# GradientGPU — Forward sensitivities on GPU (PyTorch + torchdiffeq)

This notebook is a GPU-targeted version of `NewGradient.ipynb` (same model and algorithm) with the following changes:
- Selects a device (prefers CUDA/MPS when available).
- Uses a torch-native ODE solver (`dopri5`) so integration and sensitivity propagation can run on the selected device.
- Notes: the original used a SciPy BDF solver (stiff, CPU-only). True implicit BDF integrators that run on GPU are not available in torchdiffeq; `dopri5` is explicit and may require smaller timesteps and tuning for stiff problems. If you have an AMD GPU, install a ROCm-enabled PyTorch build so CUDA-equivalent GPU acceleration is available (PyTorch for ROCm uses the same CUDA API in code).
- Keep identical model, analytic Jacobians, and optimization loop; we only adapt device and solver choice.

In [20]:
# Imports and settings (GPU-aware)
import os
import math
import numpy as np

# Set environment variables for ROCm/HIP compatibility with older GPUs
os.environ['HSA_OVERRIDE_GFX_VERSION'] = '8.0.3'  # For RX 580 (Polaris)
os.environ['AMD_SERIALIZE_KERNEL'] = '3'  # Better error reporting
os.environ['HIP_VISIBLE_DEVICES'] = '0'

try:
    import torch
    from torch import tensor
except ModuleNotFoundError as _err:
    raise ModuleNotFoundError("PyTorch is not installed in this environment.")
import matplotlib.pyplot as plt
# torchdiffeq odeint (torch-only solvers) -> runs on GPU when tensors are on device
from torchdiffeq import odeint

# dtype/device selection: prefer CUDA, then MPS (Apple), otherwise CPU
# For AMD Radeon/ROCm builds, torch.cuda.is_available() will be True when a ROCm-enabled PyTorch is installed
torch.set_default_dtype(torch.float64)
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else 'cpu'))
dtype = torch.get_default_dtype()
print(f'Using device: {device}  dtype: {dtype}')

# Solver and integration settings (tune for stability on stiff systems)
T_FINAL   = 100.0
N_TSTEPS  = 201               # sampling grid for cost/trapezoid (kept from original)
RTOL      = 1e-6
ATOL      = 1e-8
EPS_CT    = 1e-12
SEED      = 0
MAX_ITERS = 50
LR        = 1e-3
N_PROFILE = 41
PLOT_EVERY = 5

torch.manual_seed(SEED)

Using device: cuda  dtype: torch.float64


In [21]:
# -------------------------
# Model (same as original)
# -------------------------
def f_torch(state, p_vec):
    # state: tensor length 6  -> [C2, CP, pM, M, Y, YP]
    C2, CP, pM, M, Y, YP = state

    k1_aa_over_CT = p_vec[0]
    k2            = p_vec[1]
    k3_CT         = p_vec[2]
    k4            = p_vec[3]
    k4prime       = p_vec[4]
    k5_minusP     = p_vec[5]
    k6            = p_vec[6]
    k7            = p_vec[7]
    k8_minusP     = p_vec[8]
    k9            = p_vec[9]
    CT            = p_vec[10]

    denomC = CT + EPS_CT
    k3 = k3_CT / denomC
    k1 = k1_aa_over_CT * CT

    # F_M
    FM = k4prime + k4 * (M / denomC)**2

    # rates
    dC2 = k6 * M - k8_minusP * C2 + k9 * CP
    dCP = -k3 * CP * Y + k8_minusP * C2 - k9 * CP
    dpM = k3 * CP * Y - pM * FM + k5_minusP * M
    dM  = pM * FM - k5_minusP * M - k6 * M
    dY  = k1 - k2 * Y - k3 * CP * Y
    dYP = k6 * M - k7 * YP

    return torch.stack([dC2, dCP, dpM, dM, dY, dYP])

# -------------------------
# Analytical Jacobians (same as original)
# -------------------------
def Jx_f(state, p_vec):
    C2, CP, pM, M, Y, YP = state
    k1_aa_over_CT, k2, k3_CT, k4, k4prime, k5m, k6, k7, k8m, k9, CT = p_vec

    denomC = CT + EPS_CT
    k3 = k3_CT / denomC
    FM = k4prime + k4 * (M / denomC)**2
    dFM_dM = 2.0 * k4 * M / (denomC**2)

    J = torch.zeros((6,6), dtype=dtype, device=device)

    # dC2
    J[0,0] = -k8m
    J[0,1] =  k9
    J[0,3] =  k6

    # dCP
    J[1,0] =  k8m
    J[1,1] = -k3*Y - k9
    J[1,4] = -k3*CP

    # dpM
    J[2,1] =  k3*Y
    J[2,2] = -FM
    J[2,3] = -pM * dFM_dM + k5m
    J[2,4] =  k3*CP

    # dM
    J[3,2] =  FM
    J[3,3] =  pM * dFM_dM - k5m - k6

    # dY
    J[4,1] = -k3*Y
    J[4,4] = -k2 - k3*CP

    # dYP
    J[5,3] =  k6
    J[5,5] = -k7

    return J

def Jw_f(state, p_vec):
    C2, CP, pM, M, Y, YP = state
    k1aa, k2, k3_CT, k4, k4prime, k5m, k6, k7, k8m, k9, CT = p_vec

    denomC = CT + EPS_CT
    k3 = k3_CT / denomC
    FM = k4prime + k4 * (M / denomC)**2

    # helpful partials
    dk3_d_k3CT = 1.0 / denomC
    dk3_d_CT   = -k3_CT / (denomC**2)
    dk1_d_k1aa = CT
    dk1_d_CT   = k1aa
    dFM_d_k4   = (M / denomC)**2
    dFM_d_k4p  = 1.0
    dFM_d_CT   = -2.0 * k4 * (M**2) / (denomC**3)

    Jw = torch.zeros((6,11), dtype=dtype, device=device)

    # param 0: k1_aa_over_CT
    Jw[4,0] = dk1_d_k1aa                     # only dY

    # param 1: k2
    Jw[4,1] = -Y

    # param 2: k3_CT
    Jw[1,2] = -(CP*Y) * dk3_d_k3CT           # dCP
    Jw[2,2] =  (CP*Y) * dk3_d_k3CT           # dpM
    Jw[4,2] = -(CP*Y) * dk3_d_k3CT           # dY

    # param 3: k4 (via FM)
    Jw[2,3] = -pM * dFM_d_k4                 # dpM
    Jw[3,3] =  pM * dFM_d_k4                 # dM

    # param 4: k4prime
    Jw[2,4] = -pM
    Jw[3,4] =  pM

    # param 5: k5_minusP
    Jw[2,5] =  M
    Jw[3,5] = -M

    # param 6: k6
    Jw[0,6] =  M
    Jw[3,6] = -M
    Jw[5,6] =  M

    # param 7: k7
    Jw[5,7] = -YP

    # param 8: k8_minusP
    Jw[0,8] = -C2
    Jw[1,8] =  C2

    # param 9: k9
    Jw[0,9] =  CP
    Jw[1,9] = -CP

    # param 10: CT  (via k1, k3, FM)
    Jw[1,10] = -(CP*Y) * dk3_d_CT
    Jw[2,10] =  (CP*Y) * dk3_d_CT - pM * dFM_d_CT
    Jw[3,10] =  pM * dFM_d_CT
    Jw[4,10] =  dk1_d_CT - (CP*Y) * dk3_d_CT

    return Jw




In [22]:
# -------------------------
# Problem data
# -------------------------
x0 = torch.tensor([0.9, 0.05, 0.0, 0.005, 0.3, 0.0], device=device, dtype=dtype)

v1 = torch.tensor([1.5e-02, 0.0, 2.0e+02, 1.8e+02, 1.8e-02, 0.0, 1.0, 0.6, 1.0e+02, 5.0e+01, 1.0],
                  device=device, dtype=dtype)
v2 = torch.tensor([1.16000589e-02, 0.0, 5.02761655e+02, 5.74579613e+02, 1.24596408e-02, 0.0, 
                   1.64052318, 8.22972056, 1.0e+02, 5.0e+01, 1.0],
                  device=device, dtype=dtype)

n_x = x0.numel()
n_w = v1.numel()

ts = torch.linspace(0.0, T_FINAL, N_TSTEPS, device=device, dtype=dtype)

print(f'Problem dimensions: n_x={n_x}, n_w={n_w}, time grid size={N_TSTEPS}')

RuntimeError: HIP error: invalid device function
HIP kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing AMD_SERIALIZE_KERNEL=3
Compile with `TORCH_USE_HIP_DSA` to enable device-side assertions.


In [ ]:
# -------------------------
# Solvers (GPU-compatible using torchdiffeq)
# -------------------------
def solve_state(w, method='dopri5'):
    """Return X(t;w) sampled on ts (shape [N_TSTEPS, 6]).
    Uses dopri5 (explicit Runge-Kutta) which runs on GPU."""
    func = lambda t, y: f_torch(y, w)
    X = odeint(func, x0, ts, method=method, rtol=RTOL, atol=ATOL)  # [N, 6]
    return X

def solve_state_and_sensitivities(w, method='dopri5'):
    """Return X(t;w) and S(t)=dX/dw on ts.
       X shape [N,6], S shape [N,6,11].
       Solves augmented ODE: dX/dt = f(X,w), dS/dt = Jx·S + Jw"""
    # augmented state: [X, vec(S)]
    y0 = torch.cat([x0, torch.zeros(n_x*n_w, device=device, dtype=dtype)], dim=0)

    def aug_rhs(t, y):
        X = y[:n_x]
        S = y[n_x:].view(n_x, n_w)
        fX = f_torch(X, w)
        Jx = Jx_f(X, w)
        Jw = Jw_f(X, w)
        dS = Jx @ S + Jw
        return torch.cat([fX, dS.reshape(-1)])

    Y = odeint(aug_rhs, y0, ts, method=method, rtol=RTOL, atol=ATOL)  # [N, 6+6*11]
    X = Y[:, :n_x]
    S = Y[:, n_x:].view(-1, n_x, n_w)
    return X, S

In [ ]:
# -------------------------
# Helper function to compute YT_rel and M_rel from trajectory
# -------------------------
def compute_tyson_observables(X, CT):
    """Given trajectory X [N,6] and CT value, return YT_rel and M_rel.
       X columns: [C2, CP, pM, M, Y, YP]"""
    C2, CP, pM, M, Y, YP = X[:, 0], X[:, 1], X[:, 2], X[:, 3], X[:, 4], X[:, 5]
    YT = Y + YP + pM + M
    YT_rel = YT / (CT + EPS_CT)
    M_rel = M / (CT + EPS_CT)
    return YT_rel, M_rel

In [ ]:
# -------------------------
# Prepare target (Xavg from v1, v2) - integrated on selected device
# -------------------------
with torch.no_grad():
    X1 = solve_state(v1)        # [N,6]
    X2 = solve_state(v2)        # [N,6]
    WEIGHT = 1
    Xavg = (1-WEIGHT)*X1 + WEIGHT*X2
    CT_avg = (1-WEIGHT)*v1[10] + WEIGHT*v2[10]
    YT_avg, M_avg = compute_tyson_observables(Xavg, CT_avg)

print('Prepared Xavg on device:', Xavg.device)

# -------------------------
# Optimize theta (SGD on forward-sensitivity gradient) with non-negativity constraint
# -------------------------
torch.manual_seed(SEED)
theta_mid = 0.5 * (v1 + v2)
# initialize theta on device and ensure non-negative
theta     = (theta_mid + 0.2 * (v2 - v1) * torch.randn_like(v1)).clone().to(device)
theta = theta.clamp(min=0.0)

history = []

for it in range(1, MAX_ITERS+1):
    # sample t ~ U(0,1)
    t_samp = torch.rand(()).item()
    w_samp = phi_theta(theta, t_samp)

    # forward sensitivities for dC/dw
    C_samp, dC_dw = trajectory_cost_and_grad_w(w_samp, Xavg)

    # chain rule: dL/dtheta = dC/dw * dphi/dtheta
    mult = dphi_dtheta_scalar(t_samp)
    dL_dtheta = dC_dw * mult

    # gradient step (apply update and then clamp to enforce theta >= 0)
    with torch.no_grad():
        theta -= LR * dL_dtheta
        # in-place clamp to enforce non-negativity
        theta.clamp_(min=0.0)

    # (optional) track mean cost on a small grid for monitoring
    with torch.no_grad():
        t_eval = torch.linspace(0, 1, 7, device=device, dtype=dtype)
        costs = []
        for tt in t_eval:
            Ctt, _ = trajectory_cost_and_grad_w(phi_theta(theta, float(tt)), Xavg)
            costs.append(Ctt.item())
        history.append(float(np.mean(costs)))

        if it % PLOT_EVERY == 0 or it == 1:
            print(f"iter {it:03d} | sample t={t_samp:.3f} | C_samp={C_samp.item():.3e} | mean_cost~{history[-1]:.3e}")
            print(f"  theta = {theta.cpu().numpy()}")

            # Plotting trajectories (sampled point)
            X_samp = solve_state(w_samp)
            CT_samp = w_samp[10].item()
            YT_samp, M_samp = compute_tyson_observables(X_samp, CT_samp)
            t_np = ts.cpu().numpy()
            plt.figure(figsize=(9,4))
            plt.plot(t_np, YT_avg.cpu().numpy(), 'k--', lw=1.5, alpha=0.6, label='[YT]/[CT] avg (baseline)')
            plt.plot(t_np, M_avg.cpu().numpy(), 'k:', lw=1.0, alpha=0.6, label='[M]/[CT] avg (baseline)')
            plt.plot(t_np, YT_samp.cpu().numpy(), 'b-', lw=1.5, label=f'[YT]/[CT] iter {it} (t={t_samp:.3f})')
            plt.plot(t_np, M_samp.cpu().numpy(), 'r-', lw=1.0, label=f'[M]/[CT] iter {it} (t={t_samp:.3f})')
            plt.xlim(0, 100)
            plt.ylim(-0.01, 0.42)
            plt.xlabel('time (min)')
            plt.ylabel('relative concentration')
            plt.title(f'Tyson 6-ODE — Iteration {it} | Cost={C_samp.item():.3e}')
            plt.legend(fontsize=8)
            plt.grid(True)
            plt.tight_layout()
            plt.show()

# final theta
theta_opt = theta.detach().clone()
print('Final optimized theta (cpu):\n', theta_opt.cpu().numpy())

NameError: name 'v1' is not defined

In [ ]:
# -------------------------
# Cost and gradient wrt w (uses forward sensitivities)
# -------------------------
def trajectory_cost_and_grad_w(w, Xavg, method='dopri5'):
    """C(w) = ∫ 0.5||X(t;w)-Xavg(t)||^2 dt;  dC/dw = ∫ (X-Xavg)^T S dt."""
    X, S = solve_state_and_sensitivities(w, method=method)           # [N,6], [N,6,11]
    diff = X - Xavg                                   # [N,6]
    integrand = 0.5 * (diff**2).sum(dim=1)            # [N]
    C = torch.trapz(integrand, ts)

    # (X-Xavg)^T @ S -> [N, 11]
    integrand_g = torch.einsum('ni, nij -> nj', diff, S)
    dC_dw = torch.trapz(integrand_g, ts, dim=0)       # [11]
    return C, dC_dw


In [ ]:
# -------------------------
# Bezier helpers (unchanged)
# -------------------------
def phi_theta(theta, t_scalar):
    t = torch.as_tensor(t_scalar, dtype=dtype, device=device)
    return (1.0 - t)**2 * v1 + 2.0*t*(1.0 - t)*theta + t**2 * v2

def dphi_dtheta_scalar(t_scalar):
    t = torch.as_tensor(t_scalar, dtype=dtype, device=device)
    return 2.0 * t * (1.0 - t)


In [ ]:
# -------------------------
# Compare cost profiles along t (same as original)
# -------------------------
def cost_along_curve(theta_ctrl, t_grid, method='dopri5'):
    vals = []
    with torch.no_grad():
        for tt in t_grid:
            Ctt, _ = trajectory_cost_and_grad_w(phi_theta(theta_ctrl, float(tt)), Xavg, method=method)
            vals.append(Ctt.item())
    return np.array(vals)

t_plot = np.linspace(0.0, 1.0, N_PROFILE)
theta_opt  = theta_opt
torch.manual_seed(SEED + 1)
theta_rand = (theta_mid + 0.2 * (v2 - v1) * torch.randn_like(v1)).clone().to(device)
theta_rand = theta_rand.clamp(min=0.0)
theta_line = theta_mid.clone().to(device)
theta_line = theta_line.clamp(min=0.0)

cost_opt  = cost_along_curve(theta_opt,  t_plot)
cost_rand = cost_along_curve(theta_rand, t_plot)
cost_line = cost_along_curve(theta_line, t_plot)

plt.figure(figsize=(8,5))
plt.plot(t_plot, cost_opt,  label='optimized θ')
plt.plot(t_plot, cost_rand, label='random θ')
plt.plot(t_plot, cost_line, label='straight line (midpoint)')
plt.xlabel('Bezier parameter t')
plt.ylabel('C(φ_θ(t))')
plt.title('Cost along Bezier curve in parameter space')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,3))
plt.plot(np.arange(1, len(history)+1), history, '-o')
plt.xlabel('SGD iteration')
plt.ylabel('mean cost on small t-grid')
plt.title('Optimization progress (forward sensitivities)')
plt.grid(True)
plt.tight_layout()
plt.show()

## Notes & caveats
- This notebook switches the integrator to `dopri5` so the whole computation (integration and sensitivity ODE) stays in PyTorch and can run on the chosen device.
- `dopri5` is explicit; for stiff systems the original BDF may be much more stable. If you observe instability or extremely small timesteps, consider:
  - Running the original notebook on CPU with SciPy BDF for correctness comparisons.
  - Installing a ROCm-enabled PyTorch (if you have AMD Radeon) so `torch.cuda.is_available()` returns True and GPU acceleration is active.
- To install ROCm/PyTorch for AMD (high level): follow PyTorch ROCm instructions matching your kernel/driver/OS at https://pytorch.org. Once a ROCm build is installed, existing code using `device='cuda'` should run on the AMD GPU.
- If you want, I can: (1) add an option to run fixed-step RK4 with many small steps for more stable behaviour, (2) add command-line/run cells that benchmark CPU vs GPU, or (3) attempt a small smoke-run here to validate (requires executing the notebook).